In [ ]:
# Check installed package versions
import sys
from importlib.metadata import version

try:
    import langchain
    print(f"✓ langchain: {langchain.__version__}")
except:
    print("✗ langchain not installed")

try:
    import langchain_core
    print(f"✓ langchain-core: {langchain_core.__version__}")
except:
    print("✗ langchain-core not installed - REQUIRED!")
    print("  Run: pip install langchain-core")

try:
    import langchain_openai
    print(f"✓ langchain-openai: {version('langchain-openai')}")
except:
    print("✗ langchain-openai not installed")

try:
    import langchain_community
    print(f"✓ langchain-community: {langchain_community.__version__}")
except:
    print("✗ langchain-community not installed")

print(f"\nPython version: {sys.version}")
print("\nIf any packages are missing, run:")
print("pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf python-dotenv tiktoken")

In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from pathlib import Path

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#OpenAI integration for embedding and LLM
from langchain_openai import OpenAIEmbeddings,ChatOpenAI

from langchain_community.vectorstores import FAISS

#langchain core components
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
print("All import successful")

In [ ]:
#load env variable from .env file

load_dotenv

if not os.getenv("OPENAI_API_KEY"):
    print("key not found")
else:
    print("key found") 

In [ ]:
#loading pdf document

pdf_path= "pdfs/cvpaper.pdf"

if not os.path.exists(pdf_path):
    print("No pdf file exists")
else:
    loader= PyPDFLoader(pdf_path)

    #load all pages from pdf, each page is seperate document object
    documents= loader.load()

    print(f"loaded len{documents} pages from {pdf_path}")
    print(f"----Document Preview----")
    print(f"Content first(500 characters): {documents[0].page_content[:500]}...")
    print(f"Metadata:{documents[0].metadata}")
    print(f"Total characters across all page: {sum(len(doc.page_content)for doc in documents): ,}")



In [ ]:
#Initialize the testsplitte
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 1024,chunk_overlap= 128,
                                               length_function=len, separators=["\n \n", "\n", "",""])
#split the document into chunks

chunks= text_splitter.split_documents(documents)
print(f"Split {len(documents)} pages into {len(chunks)} chunks")

#preview chunks
print(f"\n---Chunks Example----")
for i,chunk in enumerate(chunks[:2]):
    print(f"\nChunk{i+1} (length: {len(chunk.page_content)} chars):")
    print(f"{chunk.page_content[:200]}...")

In [ ]:
#create embedding

embeddings= OpenAIEmbeddings(model="text-embedding-3-small")


sample_text= "This is test sentence"
sample_embedding= embeddings.embed_query(sample_text)

print("Embedding initiated")
print(f"Dimension of the embedding {len(sample_embedding)}")
